In [ ]:
!pip install pandas numpy scikit-learn xgboost shap matplotlib seaborn scipy --break-system-packages

In [ ]:
"""
Shared Modelling Pipeline
======================================
Dissertation: Predicting Revenue Growth and Cost Reduction from Business AI Adoption
 
PURPOSE
-------
Build ONE pipeline (preprocessing -> training -> SHAP), parametrised by
`target_column`, so it can be run identically for both:
    - revenue_growth_percent  (Target 1)
    - cost_reduction_percent  (Target 2)
 
The concept is build once, run twice.
It ensures both models are trained under IDENTICAL conditions (same split,
same seed, same preprocessing logic), which is essential for RQ2's
comparative SHAP analysis to be methodologically valid.
 
WHAT THIS SCRIPT DOES
----------------------
1. Loads data, documents feature selection (with justification)
2. Defines a single `run_pipeline(target_column)` function covering:
     - preprocessing (encoding + scaling, model-specific)
     - train/test split (fixed seed, stratified by company_size)
     - 5-fold cross-validation
     - training 3 models: Linear Regression, Random Forest, XGBoost
     - hyperparameter tuning (grid search) for RF and XGBoost
     - evaluation (RMSE, MAE, R²) on held-out test set
     - SHAP value computation for XGBoost (TreeExplainer)
     - saves all outputs (models, metrics, SHAP values) to disk
3. TEST RUN: executes the pipeline on revenue_growth_percent only
   (cost_reduction_percent run is a one-line call once this is validated)
 
Run: python shared_pipeline.py
"""
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import pickle
import os
import warnings
from datetime import datetime
 
warnings.filterwarnings("ignore")
 
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score, ParameterGrid
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError("Install xgboost first: pip install xgboost --break-system-packages")
 
try:
    from tqdm.auto import tqdm
except ImportError:
    # Fallback: a no-op wrapper so the script still runs if tqdm isn't installed
    def tqdm(iterable, **kwargs):
        return iterable
 
try:
    import shap
except ImportError:
    raise ImportError("Install shap first: pip install shap --break-system-packages")
    
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════
 
DATA_PATH = "ai_company_adoption.csv"   
OUT_DIR = "pipeline_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/models", exist_ok=True)
os.makedirs(f"{OUT_DIR}/shap", exist_ok=True)
os.makedirs(f"{OUT_DIR}/metrics", exist_ok=True)
 
RANDOM_SEED = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
 
# ═══════════════════════════════════════════════════════════════════════════
# FEATURE SELECTION — FINALISED AND DOCUMENTED
# ═══════════════════════════════════════════════════════════════════════════
"""
FEATURE SELECTION RATIONALE (for Methodology Section 3.5)
------------------------------------------------------------
Two target variables are modelled SEPARATELY (see Section 3.4):
    TARGET_1 = revenue_growth_percent
    TARGET_2 = cost_reduction_percent
 
The SAME feature set is used to predict both targets, so that SHAP
comparisons in RQ2 are directly comparable (same inputs, different outputs).
 
INCLUDED FEATURES, by category:
 
  AI Adoption & Investment (predictors):
    - ai_adoption_rate
    - ai_adoption_stage      (categorical: none/pilot/partial/full)
    - ai_budget_percentage
    - ai_investment_per_employee
    - ai_training_hours
    - years_using_ai
    - ai_projects_active
 
  Operational (predictors — NOT targets, see Section 2.1 Para 3 justification):
    - task_automation_rate
    - productivity_change_percent
    [time_saved_per_week EXCLUDED — see exclusion note below]
 
  Governance (predictors — feeds RQ4):
    - ai_risk_management_score
    - regulatory_compliance_score
    - ai_ethics_committee     (binary/categorical)
    - data_privacy_level
 
  Company Profile (predictors — feeds RQ3 subgroup context):
    - industry               (categorical)
    - company_size           (categorical)
    - region                 (categorical)
    - annual_revenue_usd_millions
    - num_employees
 
EXCLUDED VARIABLES:
    - time_saved_per_week
      REASON: correlation with task_automation_rate = 0.793 (confirmed via
      EDA correlation check). This level of collinearity risks inflating or
      deflating SHAP values for both features. task_automation_rate was
      retained as it shows marginally stronger correlations with BOTH
      target variables (r=0.327 with revenue_growth vs 0.291 for
      time_saved_per_week; r=0.484 with cost_reduction vs 0.401).
 
    - ai_maturity_score
      REASON: Variance Inflation Factor (VIF) analysis found severe
      multicollinearity (VIF > 200,000) between ai_maturity_score and four
      other retained predictors (ai_adoption_rate, ai_training_hours,
      ai_budget_percentage, ai_projects_active), indicating it functions as
      a near-exact composite of these variables rather than an independent
      predictor. The four component variables were retained in preference,
      as they are more specific and actionable for the study's non-technical
      decision-maker audience than a single abstract composite score.
 
NOT USED AS FEATURES (identifiers / leakage risk):
    - company_id (or similar identifier column, if present)
    - Any column that is a direct function of the target variable itself
 
Adjust FEATURE_COLS below to match your dataset's EXACT column names.
"""
 
TARGET_1 = "revenue_growth_percent"
TARGET_2 = "cost_reduction_percent"
 
NUMERIC_FEATURES = [
    "ai_adoption_rate",
    "ai_budget_percentage",
    "ai_investment_per_employee",
    "ai_training_hours",
    "years_using_ai",
    "ai_projects_active",
    "task_automation_rate",
    "productivity_change_percent",
    "ai_risk_management_score",
    "regulatory_compliance_score",
    "annual_revenue_usd_millions",
    "num_employees",
]
 
CATEGORICAL_FEATURES = [
    "ai_adoption_stage",
    "ai_ethics_committee",
    "data_privacy_level",
    "industry",
    "company_size",
    "region",
]
 
EXCLUDED_FEATURES = ["time_saved_per_week", "ai_maturity_score"]  # documented above — multicollinearity
 
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
 
# ═══════════════════════════════════════════════════════════════════════════
# LOAD DATA & VALIDATE FEATURE LIST AGAINST ACTUAL COLUMNS
# ═══════════════════════════════════════════════════════════════════════════
 
print("=" * 80)
print("LOADING DATA & VALIDATING FEATURE SELECTION")
print("=" * 80)
 
try:
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find '{DATA_PATH}'.\n"
        f"Update DATA_PATH at the top of this script to point to your CSV file."
    )
 
missing_numeric = [c for c in NUMERIC_FEATURES if c not in df.columns]
missing_categorical = [c for c in CATEGORICAL_FEATURES if c not in df.columns]
missing_targets = [t for t in [TARGET_1, TARGET_2] if t not in df.columns]
 
if missing_numeric or missing_categorical or missing_targets:
    print("⚠ WARNING — some expected columns were not found in the dataset:")
    if missing_numeric:
        print(f"  Missing numeric features: {missing_numeric}")
    if missing_categorical:
        print(f"  Missing categorical features: {missing_categorical}")
    if missing_targets:
        print(f"  Missing target columns: {missing_targets}")
    print(f"\n  Available columns in dataset:\n  {list(df.columns)}")
    print(f"\n  --> Update NUMERIC_FEATURES / CATEGORICAL_FEATURES / TARGET_1 / TARGET_2")
    print(f"      at the top of this script to match your actual column names.\n")
    # Filter to only columns that DO exist, so the script can still run for demonstration
    NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c in df.columns]
    CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in df.columns]
    ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
 
print(f"Final feature set ({len(ALL_FEATURES)} features):")
print(f"  Numeric ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")
print(f"  Categorical ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")
print(f"  Excluded (documented): {EXCLUDED_FEATURES}")
 
# Save feature documentation to file (for Methodology chapter reference)
with open(f"{OUT_DIR}/feature_selection_documentation.txt", "w") as f:
    f.write("FEATURE SELECTION — FINALISED\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Numeric features ({len(NUMERIC_FEATURES)}):\n")
    for feat in NUMERIC_FEATURES:
        f.write(f"  - {feat}\n")
    f.write(f"\nCategorical features ({len(CATEGORICAL_FEATURES)}):\n")
    for feat in CATEGORICAL_FEATURES:
        f.write(f"  - {feat}\n")
    f.write(f"\nExcluded features:\n")
    exclusion_reasons = {
        "time_saved_per_week": "multicollinearity with task_automation_rate, r=0.793",
        "ai_maturity_score": "severe multicollinearity (VIF > 200,000) with ai_adoption_rate, "
                              "ai_training_hours, ai_budget_percentage, and ai_projects_active",
    }
    for feat in EXCLUDED_FEATURES:
        reason = exclusion_reasons.get(feat, "excluded — see documentation above")
        f.write(f"  - {feat} ({reason})\n")
    f.write(f"\nTarget variables:\n  - {TARGET_1}\n  - {TARGET_2}\n")
print(f"\nSaved: {OUT_DIR}/feature_selection_documentation.txt")
 
 
# ═══════════════════════════════════════════════════════════════════════════
# SHARED PIPELINE FUNCTION — PARAMETRISED BY target_column
# ═══════════════════════════════════════════════════════════════════════════
 
def build_preprocessor(numeric_features, categorical_features, model_type):
    """
    Build a preprocessing ColumnTransformer.

    model_type: 'linear' -> one-hot encode categoricals, scale numerics
                'tree'   -> one-hot encode categoricals, no scaling
                           (tree-based models are scale-invariant)
    """
    if model_type == "linear":
        numeric_transformer = StandardScaler()
        categorical_transformer = OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)
    else:  # tree-based
        numeric_transformer = "passthrough"
        categorical_transformer = OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)
 
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )
    return preprocessor
 
 
def get_feature_names_out(preprocessor, numeric_features, categorical_features):
    """Extract readable feature names after preprocessing (for SHAP labelling)."""
    try:
        cat_encoder = preprocessor.named_transformers_["cat"]
        cat_names = list(cat_encoder.get_feature_names_out(categorical_features))
    except Exception:
        cat_names = categorical_features
    return list(numeric_features) + cat_names
 
 
def run_pipeline(target_column, df, numeric_features, categorical_features,
                  random_seed=RANDOM_SEED, test_size=TEST_SIZE, cv_folds=CV_FOLDS,
                  out_dir=OUT_DIR, verbose=True, sample_frac=None):
    """
    Run the FULL modelling pipeline for a single target variable.
 
    Steps: preprocess -> split -> train (LR, RF, XGB) -> tune (RF, XGB) ->
           evaluate -> SHAP (XGB) -> save everything to disk.
 
    Parameters
    ----------
    sample_frac : float or None
        If set (e.g. 0.05 for 5%), the dataset is randomly downsampled to
        this fraction BEFORE any processing. Use this for a fast staged
        validation run to catch bugs/errors cheaply, before committing to
        the full-scale run. Set to None (default) for the real, full run
        used in your actual dissertation results.
 
    Returns a dict of results for downstream comparison (RQ2).
    """
    # SAFETY: quick-test runs write to files with a "_QUICKTEST" suffix
    output_column_name = target_column
    if sample_frac is not None:
        output_column_name = f"{target_column}_QUICKTEST"
 
    if verbose:
        print("\n" + "=" * 80)
        mode_label = f"QUICK TEST RUN ({sample_frac*100:.0f}% sample)" if sample_frac else "FULL RUN"
        print(f"RUNNING PIPELINE — TARGET: {target_column}  [{mode_label}]")
        print("=" * 80)
 
    t0 = datetime.now()
 
    # ── 0. Optional downsampling for fast staged validation ─────────────
    if sample_frac is not None:
        df = df.sample(frac=sample_frac, random_state=random_seed).reset_index(drop=True)
        if verbose:
            print(f"⚠ Running on a {sample_frac*100:.0f}% SAMPLE ({len(df):,} rows) for quick validation.")
            print(f"  This checks the pipeline runs end-to-end without errors — NOT for final results.")
            print(f"  Re-run with sample_frac=None once this completes successfully.\n")
 
    # ── 1. Prepare data ──────────────────────────────────────────────────
    model_df = df[numeric_features + categorical_features + [target_column]].dropna()
    if verbose:
        print(f"Rows after dropping missing values: {len(model_df):,} "
              f"(dropped {len(df) - len(model_df):,})")
 
    X = model_df[numeric_features + categorical_features]
    y = model_df[target_column]
 
    # Stratify by company_size if present (keeps subgroup distribution similar
    # across train/test — relevant for RQ3 subgroup analysis)
    stratify_col = model_df["company_size"] if "company_size" in model_df.columns else None
 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_seed, stratify=stratify_col
    )
    if verbose:
        print(f"Train set: {len(X_train):,} rows | Test set: {len(X_test):,} rows")
 
    results = {"target": target_column, "output_label": output_column_name,
               "models": {}, "n_train": len(X_train), "n_test": len(X_test)}
 
    # ── 2. LINEAR REGRESSION ─────────────────────────────────────────────
    if verbose:
        print("\n--- Training Linear Regression (baseline) ---")
 
    preproc_linear = build_preprocessor(numeric_features, categorical_features, "linear")
    lr_pipeline = Pipeline([
        ("preprocessor", preproc_linear),
        ("model", LinearRegression())
    ])
    
    shared_cv = KFold(n_splits=cv_folds, shuffle=True, random_state=random_seed)
    lr_fold_scores = -cross_val_score(
        lr_pipeline, X_train, y_train, cv=shared_cv,
        scoring="neg_root_mean_squared_error", n_jobs=-1)
    results["fold_scores"] = {"LinearRegression": list(lr_fold_scores)}

    lr_pipeline.fit(X_train, y_train)
    lr_pred = lr_pipeline.predict(X_test)
 
    lr_metrics = {
        "RMSE": round(np.sqrt(mean_squared_error(y_test, lr_pred)), 4),
        "MAE": round(mean_absolute_error(y_test, lr_pred), 4),
        "R2": round(r2_score(y_test, lr_pred), 4),
    }
    results["models"]["LinearRegression"] = lr_metrics
    if verbose:
        print(f"  RMSE={lr_metrics['RMSE']}  MAE={lr_metrics['MAE']}  R²={lr_metrics['R2']}")
 
    # ── 3. RANDOM FOREST (with grid search) ──────────────────────────────
    if verbose:
        print("\n--- Training Random Forest (grid search, 5-fold CV) ---")
 
    preproc_tree = build_preprocessor(numeric_features, categorical_features, "tree")
    rf_pipeline = Pipeline([
        ("preprocessor", preproc_tree),
        ("model", RandomForestRegressor(random_state=random_seed, n_jobs=1))
    ])
 
    rf_param_grid = {
        "model__n_estimators": [100],
        "model__max_depth": [10, 20],
        "model__min_samples_split": [2, 5],
    }
 
    rf_param_list = list(ParameterGrid(rf_param_grid))
    rf_cv = shared_cv
 
    if verbose:
        print(f"  Testing {len(rf_param_list)} hyperparameter combinations "
              f"({len(rf_param_list) * cv_folds} total model fits)...")
 
    rf_search_results = []
    for params in tqdm(rf_param_list, desc="  Random Forest grid search", disable=not verbose):
        rf_pipeline.set_params(**params)
        scores = cross_val_score(
            rf_pipeline, X_train, y_train, cv=rf_cv,
            scoring="neg_root_mean_squared_error", n_jobs=-1
        )
        rf_search_results.append({"params": params, "mean_score": scores.mean(),
                                  "fold_scores": list(scores)})

    rf_best_result = max(rf_search_results, key=lambda r: r["mean_score"])
    results["fold_scores"]["RandomForest"] = [-s for s in rf_best_result["fold_scores"]]
    pd.DataFrame([
        {**r["params"], "mean_cv_rmse": -r["mean_score"]}
        for r in rf_search_results
    ]).sort_values("mean_cv_rmse").to_csv(
        f"{out_dir}/metrics/{output_column_name}_RandomForest_grid_search.csv", index=False)
    rf_best_params = rf_best_result["params"]
 
    rf_pipeline.set_params(**rf_best_params)
    rf_pipeline.fit(X_train, y_train)
    rf_best = rf_pipeline
    rf_pred = rf_best.predict(X_test)
 
    rf_metrics = {
        "RMSE": round(np.sqrt(mean_squared_error(y_test, rf_pred)), 4),
        "MAE": round(mean_absolute_error(y_test, rf_pred), 4),
        "R2": round(r2_score(y_test, rf_pred), 4),
        "best_params": rf_best_params,
    }
    results["models"]["RandomForest"] = rf_metrics
    if verbose:
        print(f"  RMSE={rf_metrics['RMSE']}  MAE={rf_metrics['MAE']}  R²={rf_metrics['R2']}")
        print(f"  Best params: {rf_metrics['best_params']}")
 
    # ── 4. XGBOOST (with grid search) — PRIMARY MODEL ────────────────────
    if verbose:
        print("\n--- Training XGBoost (grid search, 5-fold CV) — PRIMARY MODEL ---")
 
    xgb_pipeline = Pipeline([
        ("preprocessor", preproc_tree),
        ("model", XGBRegressor(random_state=random_seed, n_jobs=-1,
                                objective="reg:squarederror"))
    ])
 
    xgb_param_grid = {
        "model__n_estimators": [100, 200],
        "model__max_depth": [4, 6, 8],
        "model__learning_rate": [0.05, 0.1],
        "model__subsample": [0.8, 1.0],
    }
 
    xgb_param_list = list(ParameterGrid(xgb_param_grid))
    xgb_cv = shared_cv
 
    if verbose:
        print(f"  Testing {len(xgb_param_list)} hyperparameter combinations "
              f"({len(xgb_param_list) * cv_folds} total model fits)...")
 
    xgb_search_results = []
    for params in tqdm(xgb_param_list, desc="  XGBoost grid search", disable=not verbose):
        xgb_pipeline.set_params(**params)
        scores = cross_val_score(
            xgb_pipeline, X_train, y_train, cv=xgb_cv,
            scoring="neg_root_mean_squared_error", n_jobs=-1
        )
        xgb_search_results.append({"params": params, "mean_score": scores.mean(),
                                   "fold_scores": list(scores)})

    xgb_best_result = max(xgb_search_results, key=lambda r: r["mean_score"])
    results["fold_scores"]["XGBoost"] = [-s for s in xgb_best_result["fold_scores"]]
    pd.DataFrame([
        {**r["params"], "mean_cv_rmse": -r["mean_score"]}
        for r in xgb_search_results
    ]).sort_values("mean_cv_rmse").to_csv(
        f"{out_dir}/metrics/{output_column_name}_XGBoost_grid_search.csv", index=False)
    xgb_best_params = xgb_best_result["params"]
 
    xgb_pipeline.set_params(**xgb_best_params)
    xgb_pipeline.fit(X_train, y_train)
    xgb_best = xgb_pipeline
    xgb_pred = xgb_best.predict(X_test)
 
    xgb_metrics = {
        "RMSE": round(np.sqrt(mean_squared_error(y_test, xgb_pred)), 4),
        "MAE": round(mean_absolute_error(y_test, xgb_pred), 4),
        "R2": round(r2_score(y_test, xgb_pred), 4),
        "best_params": xgb_best_params,
    }
    results["models"]["XGBoost"] = xgb_metrics
    if verbose:
        print(f"  RMSE={xgb_metrics['RMSE']}  MAE={xgb_metrics['MAE']}  R²={xgb_metrics['R2']}")
        print(f"  Best params: {xgb_metrics['best_params']}")
 
    # ── 5. NATIVE FEATURE IMPORTANCE (sanity check) ──────────────────────
    if verbose:
        print("\n--- Native XGBoost feature importance (sanity check) ---")
 
    xgb_model_only = xgb_best.named_steps["model"]
    feature_names = get_feature_names_out(
        xgb_best.named_steps["preprocessor"], numeric_features, categorical_features
    )
    importances = xgb_model_only.feature_importances_
    importance_df = pd.DataFrame({
        "feature": feature_names, "importance": importances
    }).sort_values("importance", ascending=False)
 
    top10 = importance_df.head(10)
    if verbose:
        print(top10.to_string(index=False))
 
    importance_df.to_csv(f"{out_dir}/metrics/{output_column_name}_native_feature_importance.csv", index=False)
 
    # ── 6. SHAP VALUES (TreeExplainer on XGBoost) ────────────────────────
    if verbose:
        print("\n--- Computing SHAP values (TreeExplainer) ---")
 
    X_test_transformed = xgb_best.named_steps["preprocessor"].transform(X_test)
    X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names)
 
    explainer = shap.TreeExplainer(xgb_model_only)
 
    # Compute SHAP values in batches with a progress bar (TreeExplainer doesn't
    # expose per-row progress natively, so we batch manually for visibility on large test sets).
    batch_size = 2000
    n_rows = len(X_test_transformed_df)
    shap_batches = []
    for start in tqdm(range(0, n_rows, batch_size), desc="  SHAP values", disable=not verbose):
        end = min(start + batch_size, n_rows)
        batch_values = explainer.shap_values(X_test_transformed_df.iloc[start:end])
        shap_batches.append(batch_values)
    shap_values = np.concatenate(shap_batches, axis=0)
 
    # Save SHAP values + the transformed test set they correspond to
    with open(f"{out_dir}/shap/{output_column_name}_shap_values.pkl", "wb") as f:
        pickle.dump({
            "shap_values": shap_values,
            "X_test_transformed": X_test_transformed_df,
            "feature_names": feature_names,
            "expected_value": explainer.expected_value,
        }, f)
    if verbose:
        print(f"  SHAP values computed for {X_test_transformed_df.shape[0]:,} test observations, "
              f"{len(feature_names)} features")
        print(f"  Saved: {out_dir}/shap/{output_column_name}_shap_values.pkl")
 
    # SHAP summary plot (beeswarm) — global importance
    fig = plt.figure(figsize=(9, 7))
    shap.summary_plot(shap_values, X_test_transformed_df, show=False, max_display=15)
    plt.title(f"SHAP Summary — {output_column_name}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/shap/{output_column_name}_shap_summary.png", dpi=140, bbox_inches="tight")
    plt.close()
    if verbose:
        print(f"  Saved: {out_dir}/shap/{output_column_name}_shap_summary.png")
 
    # ── 7. Save trained models ───────────────────────────────────────────
    with open(f"{out_dir}/models/{output_column_name}_linear_regression.pkl", "wb") as f:
        pickle.dump(lr_pipeline, f)
    with open(f"{out_dir}/models/{output_column_name}_random_forest.pkl", "wb") as f:
        pickle.dump(rf_best, f)
    with open(f"{out_dir}/models/{output_column_name}_xgboost.pkl", "wb") as f:
        pickle.dump(xgb_best, f)
 
    # ── 8. Save metrics summary ───────────────────────────────────────────
    with open(f"{out_dir}/metrics/{output_column_name}_metrics.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
 
    elapsed = (datetime.now() - t0).total_seconds()
    results["runtime_seconds"] = round(elapsed, 1)
    if verbose:
        print(f"\nPipeline complete for '{target_column}' in {elapsed:.1f}s")
        print(f"All outputs saved under: {out_dir}/")
 
    return results
 
 
# ═══════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION — set QUICK_TEST = True first to validate the pipeline
# runs end-to-end without errors on a small sample (fast — a few minutes).
# Once that succeeds, set QUICK_TEST = False and re-run for real results.
# ═══════════════════════════════════════════════════════════════════════════
QUICK_TEST = False          # <-- flip to False for the real, full-scale run
QUICK_TEST_SAMPLE_FRAC = 0.03   # 3% of 150,000 rows ≈ 4,500 rows — fast
 
if __name__ == "__main__":
 
    sample_frac = QUICK_TEST_SAMPLE_FRAC if QUICK_TEST else None
 
    print("\n" + "#" * 80)
    if QUICK_TEST:
        print(f"# QUICK TEST RUN — validating pipeline on TARGET 1 (revenue_growth_percent)")
        print(f"# Using a {QUICK_TEST_SAMPLE_FRAC*100:.0f}% sample for speed. This is NOT your final result.")
    else:
        print("# FULL RUN — Pipeline on TARGET 1 (revenue_growth_percent)")
    print("#" * 80)
 
    results_target1 = run_pipeline(
        target_column=TARGET_1,
        df=df,
        numeric_features=NUMERIC_FEATURES,
        categorical_features=CATEGORICAL_FEATURES,
        sample_frac=sample_frac,
    )
 
    # ── Summary table for RQ1 (this target only, for now) ────────────────
    t1_label = results_target1["output_label"]
    print("\n" + "=" * 80)
    print(f"SUMMARY — Model comparison for {t1_label}")
    print("=" * 80)
    summary_rows = []
    for model_name, metrics in results_target1["models"].items():
        summary_rows.append({
            "Model": model_name,
            "RMSE": metrics["RMSE"],
            "MAE": metrics["MAE"],
            "R2": metrics["R2"],
        })
    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))
    summary_df.to_csv(f"{OUT_DIR}/metrics/{t1_label}_model_comparison_table.csv", index=False)
    print(f"\nSaved: {OUT_DIR}/metrics/{t1_label}_model_comparison_table.csv")
 
    print("\n" + "#" * 80)
    if QUICK_TEST:
        print("# QUICK TEST COMPLETE.")
        print("# If this ran without errors and the results look sensible (no NaNs,")
        print("# no negative R² that's wildly off, no crashes), your pipeline is validated.")
        print("#")
        print("# NEXT STEP: open this script, set QUICK_TEST = False near the bottom,")
        print("# then re-run for the real, full-scale result (this will take longer —")
        print("# see the runtime printed above as a rough guide, scaled up for 150,000 rows).")
        print("#" * 80)
    else:
        print("# FULL RUN COMPLETE for Target 1 (revenue_growth_percent).")
        print("# Now running Target 2 (cost_reduction_percent) — same pipeline, new target.")
        print("#" * 80)
 
        results_target2 = run_pipeline(
            target_column=TARGET_2,
            df=df,
            numeric_features=NUMERIC_FEATURES,
            categorical_features=CATEGORICAL_FEATURES,
            sample_frac=None,
        )
 
        t2_label = results_target2["output_label"]
        print("\n" + "=" * 80)
        print(f"SUMMARY — Model comparison for {t2_label}")
        print("=" * 80)
        summary_rows_t2 = []
        for model_name, metrics in results_target2["models"].items():
            summary_rows_t2.append({
                "Model": model_name,
                "RMSE": metrics["RMSE"],
                "MAE": metrics["MAE"],
                "R2": metrics["R2"],
            })
        summary_df_t2 = pd.DataFrame(summary_rows_t2)
        print(summary_df_t2.to_string(index=False))
        summary_df_t2.to_csv(f"{OUT_DIR}/metrics/{t2_label}_model_comparison_table.csv", index=False)
        print(f"\nSaved: {OUT_DIR}/metrics/{t2_label}_model_comparison_table.csv")
 
        print("\n" + "#" * 80)
        print("# BOTH TARGETS COMPLETE — 'run twice' fully executed.")
        print("# You now have model comparison tables, SHAP values, and SHAP summary")
        print("# plots for both revenue_growth_percent and cost_reduction_percent,")
        print("# ready for the RQ2 comparative SHAP analysis.")
        print("#" * 80)